In [ ]:
!pip install tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/MINIPROJECT/yamnet_tf_model

__init__.py  model.py  __pycache__  README.txt	weights.h5


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/MINIPROJECT/yamnet_tf_model")

In [ ]:
import model

In [ ]:
net = model.create_model()
for layer in net.layers:
    layer.trainable = True

In [ ]:
net.load_weights(
"/content/drive/MyDrive/MINIPROJECT/yamnet_tf_model/weights.h5",
by_name=True,
skip_mismatch=True
)

In [ ]:
net.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_ (Conv2D)                │ (None, 48, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ b_ (BatchNormalization)         │ (None, 48, 32, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 48, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_               │ (None, 48, 32, 32)     │           320 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ L11_ (BatchNormalization)       │ (None, 48, 32, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 48, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1_ (Conv2D)              │ (None, 48, 32, 64)     │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ L12_ (BatchNormalization)       │ (None, 48, 32, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 48, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1_             │ (None, 24, 16, 64)     │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ L21_ (BatchNormalization)       │ (None, 24, 16, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 24, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2_ (Conv2D)              │ (None, 24, 16, 128)    │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ L22_ (BatchNormalization)       │ (None, 24, 16, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 24, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2_             │ (None, 24, 16, 128)    │         1,280 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ L31_ (BatchNormalization)       │ (None, 24, 16, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 24, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3_ (Conv2D)              │ (None, 24, 16, 128)    │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ L32_ (BatchNormalization)       │ (None, 24, 16, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_6 (ReLU)                  │ (None, 24, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_3_             │ (None, 12, 8, 128)     │         1,280 │
│ (DepthwiseConv2D)               │                        │             

 Total params: 3,241,282 (12.36 MB)

 Trainable params: 3,219,394 (12.28 MB)

 Non-trainable params: 21,888 (85.50 KB)

In [ ]:
import os
import numpy as np
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from scipy.signal import butter, filtfilt
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split


In [ ]:
def create_spectrogram_fragments(signal):

    spec = np.abs(librosa.stft(
        signal,
        n_fft=128,
        win_length=50,
        hop_length=20
    ))


    spec = np.log1p(spec)

    fragment_frames = 150
    overlap = 0.5

    step = int(fragment_frames * (1 - overlap))

    fragments = []

    for start in range(0, spec.shape[1] - fragment_frames + 1, step):

        frag = spec[:, start:start + fragment_frames]

        frag = tf.image.resize(
            frag[..., np.newaxis],
            (96, 64)
        ).numpy()
        frag = frag / np.max(frag)

        fragments.append(frag)

    return fragments

In [ ]:
from scipy.signal import butter, filtfilt
def bandpass_filter(signal, sr, low=25, high=600, order=10):

    nyquist = 0.5 * sr
    low = low / nyquist
    high = high / nyquist

    b, a = butter(order, [low, high], btype='band')

    filtered = filtfilt(b, a, signal)

    return filtered

In [ ]:
def preprocess_signal(path):

    try:
        signal, sr = librosa.load(path, sr=None)
    except Exception as e:
        print("Error loading:", path)
        print(e)
        return None

    print("Loaded:", path)

    if sr != 2000:
        signal = librosa.resample(signal, orig_sr=sr, target_sr=2000)
        sr = 2000

    signal = bandpass_filter(signal, sr)

    std = np.std(signal)

    if std == 0:
        print("STD ZERO:", path)
        return None

    signal = (signal - np.mean(signal)) / std

    duration = len(signal) / sr
    print("Duration:", duration)

    if duration < 1.6:
        print("Too short:", path)
        return None

    if duration > 10:
        signal = signal[:10 * sr]

    return signal.astype(np.float32)

In [ ]:
def load_dataset(dataset_path):

    X = []
    y = []

    class_names = ["normal", "abnormal"]

    for label, class_name in enumerate(class_names):

        folder = os.path.join(dataset_path, class_name)

        for file in os.listdir(folder):

            if file.endswith(".wav"):

                path = os.path.join(folder, file)

                signal = preprocess_signal(path)

                if signal is None:
                    continue

                fragments = create_spectrogram_fragments(signal)

                for frag in fragments:
                  X.append(frag)
                  y.append(label)

                # progress print
                if len(X) % 100 == 0:
                    print("Processed:", len(X))

    return np.array(X), np.array(y)

train_path = "/content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train"
test_path  = "/content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/test"

X_train, y_train = load_dataset(train_path)
X_test, y_test = load_dataset(test_path)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


from sklearn.model_selection import train_test_split #training the

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)




Streaming output truncated to the last 5000 lines.
Duration: 8.0
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/e00684.wav
Duration: 26.765
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/b0132.wav
Duration: 8.0
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/e01507.wav
Duration: 17.992
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/b0383.wav
Duration: 8.0
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/e01997.wav
Duration: 14.043
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/e00808.wav
Duration: 12.645
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/e00413.wav
Duration: 34.712
Loaded: /content/drive/MyDrive/MINIPROJECT/physionet_split/physionet_split/train/abnormal/e00913.wav
Duration: 17.729

In [ ]:
for layer in net.layers:
    layer.trainable = True

In [ ]:
noise = np.random.normal(0, 0.01, X_train.shape)

X_train_aug = np.concatenate([X_train, X_train + noise])
y_train_aug = np.concatenate([y_train, y_train])

In [ ]:
net.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = net.fit(
    X_train_final,
    y_train_final,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,

)

Epoch 1/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 100s 74ms/step - accuracy: 0.7888 - loss: 0.5026 - val_accuracy: 0.5762 - val_loss: 0.6805
Epoch 2/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.8023 - loss: 0.4147 - val_accuracy: 0.8026 - val_loss: 0.3861
Epoch 3/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8234 - loss: 0.3509 - val_accuracy: 0.8160 - val_loss: 0.3505
Epoch 4/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8348 - loss: 0.3186 - val_accuracy: 0.8237 - val_loss: 0.3403
Epoch 5/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8498 - loss: 0.2928 - val_accuracy: 0.8288 - val_loss: 0.3178
Epoch 6/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.8622 - loss: 0.2710 - val_accuracy: 0.8269 - val_loss: 0.3212
Epoch 7/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.8780 - loss: 0.2481 - val_accuracy: 0.8351 - val_loss: 0.3635
Epoch 8/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.8915 - loss: 0.2268 -

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, matthews_corrcoef, cohen_kappa_score

y_pred = net.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)


237/237 ━━━━━━━━━━━━━━━━━━━━ 14s 47ms/step


In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1006  565]
 [ 575 5423]]


In [ ]:
TP = np.diag(cm)
FP = np.sum(cm, axis=0) - TP
FN = np.sum(cm, axis=1) - TP
TN = np.sum(cm) - (TP + FP + FN)

In [ ]:
accuracy = np.sum(TP) / np.sum(cm)

sensitivity = np.mean(TP / (TP + FN))
specificity = np.mean(TN / (TN + FP))

precision = np.mean(TP / (TP + FP))

f1 = np.mean(2 * (precision * sensitivity) / (precision + sensitivity))

snr = np.mean(TP / (FP + 1e-10))

mcc = matthews_corrcoef(y_test, y_pred)

kappa = cohen_kappa_score(y_test, y_pred)

In [ ]:
print(f"Overall Accuracy: {round(accuracy * 100)}%")
print("Sensitivity:", sensitivity)
print("Specificity:", specificity)
print("Precision:", precision)
print("F1 Score:", f1)
print("SNR:", snr)
print("MCC:", mcc)
print("Kappa:", kappa)

Overall Accuracy: 85%
Sensitivity: 0.7722455862117417
Specificity: 0.7722455862117417
Precision: 0.7709753789679296
F1 Score: 0.7716099598434245
SNR: 5.673897652942438
MCC: 0.5432194801222535
Kappa: 0.5432151193759487


In [ ]:
from sklearn.metrics import recall_score

uar = recall_score(y_test, y_pred, average='macro')
print(f"Overall Accuracy: {round(accuracy * 100)}%")
print(f"UAR: {uar*100:.2f}")


Overall Accuracy: 85%
UAR: 77.22


In [ ]:
import sys
sys.path.append('/content/model')

from model import create_model

In [ ]:
net.save("final_model.keras")

In [ ]:
net.save("final_model.keras")
from google.colab import files
files.download("final_model.keras")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>